# morning-gemma: 공식 per_layer_embedder 재사용 변환 (Kaggle CPU)

메모리가 부족해 실패하던 `per_layer_embedder` 변환은 건너뛰고, 공식 `gemma-4-E2B-it.litertlm`에 들어 있는 것을 그대로 넣는다.
LoRA는 attention·MLP 층에만 붙였으므로 이 단어 표(`embed_tokens_per_layer`)는 원본과 같아야 한다 → 2번 칸에서 숫자로 확인.

1. 준비  2. 공식 파일 풀기  3. 변환(PLE만 공식 것, PLE 표는 메모리에서 지움, prefill은 128 하나만)  4. 신호 이름 비교  5. 대답 확인  6. (확인용) 단어 표가 원본과 같은지

In [ ]:
import os, glob, subprocess
MERGED = os.path.dirname(glob.glob("/kaggle/input/**/config.json", recursive=True)[0])
print("merged:", MERGED); print(sorted(os.listdir(MERGED)))
!free -g; nproc; df -h /kaggle/working /tmp | tail -2
!pip install -q uv
!uv tool install -q litert-torch-nightly
!uv tool install -q litert-lm
os.environ["PATH"] += ":/root/.local/bin"
PY = subprocess.check_output(["uv", "tool", "dir"], text=True).strip() + "/litert-torch-nightly/bin/python"
print("PY:", PY); assert os.path.exists(PY)
# (앞서 V2를 시험하느라 tf-nightly를 넣었던 환경을 원래대로 되돌린다)
!{PY} -c "import tensorflow" 2>/dev/null && uv tool install -q --force litert-torch-nightly
!{PY} -c "import tensorflow" 2>/dev/null && echo 'tf 남아 있음' || echo '깨끗한 환경'

In [ ]:
%%writefile /tmp/inspect.py
# .litertlm을 풀어서 구성 파일과 각 tflite의 신호(signature) 이름을 보여 준다.
import sys, os, json
from litert_torch.generative.export_hf.experimental.litertlm_bundle import litertlm_bundle as L
src, dst = sys.argv[1], sys.argv[2]
comp = L.unpack_litertlm(src, dst)
sigs = {}
for k, v in comp.items():
    size = os.path.getsize(v) if isinstance(v, str) and os.path.isfile(v) else ""
    print(f"{k:32s} {size:>12} {v}")
    if isinstance(v, str) and v.endswith(".tflite"):
        try:
            from ai_edge_litert.interpreter import Interpreter
            sl = Interpreter(model_path=v).get_signature_list()
            sigs[k] = {n: sorted(d["inputs"]) for n, d in sl.items()}
        except Exception as e:
            sigs[k] = f"(signature 읽기 실패: {e})"
json.dump({"components": {k: str(v) for k, v in comp.items()}, "sigs": sigs}, open(dst + "/summary.json", "w"), indent=1, ensure_ascii=False)
for k, s in sigs.items():
    print("==", k); print(json.dumps(s, ensure_ascii=False)[:1500])


In [ ]:
!{PY} -c "from huggingface_hub import hf_hub_download as d; print(d('litert-community/gemma-4-E2B-it-litert-lm','gemma-4-E2B-it.litertlm',local_dir='/tmp/official'))"
if not os.path.exists("/tmp/official_unpacked/summary.json"):
    !rm -rf /tmp/official_unpacked && {PY} /tmp/inspect.py /tmp/official/gemma-4-E2B-it.litertlm /tmp/official_unpacked 2>&1 | grep -v Warning | tail -40
import json
OFFICIAL_PLE = json.load(open("/tmp/official_unpacked/summary.json"))["components"]["tf_lite_per_layer_embedder"]
print("OFFICIAL_PLE:", OFFICIAL_PLE)

In [ ]:
%%writefile /tmp/export_reuse_ple.py
# per_layer_embedder 변환 단계만 "공식 파일 경로 끼워 넣기"로 바꾸고 나머지는 원래 도구 그대로 돌린다.
import sys, dataclasses
merged, official_ple = sys.argv[1], sys.argv[2]
from litert_torch.generative.export_hf.core import export_lib
def reuse_official_ple(source_model_artifacts, export_config, exported_model_artifacts):
    print(">>> per_layer_embedder: 변환 건너뜀, 공식 파일 사용:", official_ple, flush=True)
    # 본체(prefill/decode)는 이 단어 표를 입력(per_layer_embeddings)으로 받으므로 표 자체는 필요 없다.
    # 32비트로 약 9.4GB라서 본체 변환 전에 메모리에서 지운다.
    import gc, torch
    for name, mod in source_model_artifacts.model.named_modules():
        if name.endswith("embed_tokens_per_layer") and hasattr(mod, "weight"):
            n = mod.weight.numel()
            mod.weight = torch.nn.Parameter(torch.empty(0, mod.weight.shape[1]), requires_grad=False)
            print(f">>> 메모리에서 지움: {name} ({n * 4 / 1e9:.1f}GB)", flush=True)
    gc.collect()
    paths = dict(exported_model_artifacts.additional_model_paths or {})
    paths["per_layer_embedder"] = official_ple
    return dataclasses.replace(exported_model_artifacts, additional_model_paths=paths)
export_lib.export_additional_models = reuse_official_ple
# 11/21 3차: 공식 PLE 출력은 "표 값 그대로"(배율 1)이고, 우리 본체는 "표 x sqrt(256)=16"을 기대한다
# (점검 칸: 공식 출력 std 0.062 vs 표x16 std 1.07, 층별 상관 0.8~0.94). 그대로 두면 대답이 이모지로 깨진다.
# → 본체(prefill/decode)가 받은 per_layer_embeddings에 sqrt(d)를 곱하게 한다.
import inspect, json as _json, math
from litert_torch.generative.export_hf.model_ext.gemma4 import exportable_module as g4
_cfg = _json.load(open(merged + "/config.json")); _tc = _cfg.get("text_config", _cfg)
PLE_SCALE = math.sqrt(_tc["hidden_size_per_layer_input"])
def _scaled(orig):
    def adapt_inputs(self, tokens, embeddings, per_layer_embeddings, *a, **k):
        return orig(self, tokens, embeddings, per_layer_embeddings * PLE_SCALE, *a, **k)
    return adapt_inputs
for _n, _c in list(vars(g4).items()):
    if inspect.isclass(_c) and "adapt_inputs" in _c.__dict__ and "per_layer_embeddings" in inspect.signature(_c.__dict__["adapt_inputs"]).parameters:
        _c.adapt_inputs = _scaled(_c.__dict__["adapt_inputs"]); print(f">>> PLE x{PLE_SCALE} 패치: {_n}", flush=True)
from litert_torch.generative.export_hf import export as lib
# 앱은 글자만 쓰므로 그림(vision)·소리(audio) 인코더는 만들지 않는다.
# (use_v2=True는 메모리가 4GB로 줄었지만 tf-nightly와 버전이 안 맞아 MLIR 오류가 났다 → 기존 변환기 사용)
lib.export(model=merged, output_dir="out", externalize_embedder=True,
           jinja_chat_template_override="litert-community/gemma-4-E2B-it-litert-lm",
           export_vision_encoder=False, export_audio_encoder=False,
           # 11/21: 31GB에서 마지막 "Merge MLIR Modules"(prefill_128+prefill_1024+decode 합치기) 중 메모리 부족.
           # → prefill_1024를 빼서 합칠 덩어리를 줄인다. 긴 입력은 런타임이 128씩 나눠 넣는다(NPU 설정도 128 하나만 씀).
           # → 캐시 길이도 32771 → 4096. 앱 런타임은 원래 4096 토큰까지만 쓰므로(max_num_tokens 4096) 손해 없음.
           prefill_lengths=[128], cache_length=4096,
           # 11/21 2차: prefill_1024를 빼도 Merge 직후 29GB로 실패 → 가중치를 MLIR에 복사하지 않는 방식.
           # (11/20엔 이 옵션이 PLE 파일 쓰기에서 실패했지만, 지금은 PLE를 공식 파일로 건너뛴다)
           experimental_lightweight_conversion=True)


In [ ]:
# 메모리 사용량을 1분마다 /tmp/mem.log에 남긴다 (어디서 메모리가 모자라는지 보려고)
subprocess.Popen("while true; do date +%T; free -g | sed -n 2p; sleep 60; done > /tmp/mem.log 2>&1", shell=True)
!rm -rf out && {PY} /tmp/export_reuse_ple.py {MERGED} {OFFICIAL_PLE} 2>&1 | grep --line-buffered -E "FAIL|Error|error|>>>|Traceback|\(\+[0-9]"
!ls -la out; tail -5 /tmp/mem.log

In [ ]:
# 우리 파일과 공식 파일의 신호 이름을 비교한다 (prefill_decode가 PLE 출력을 같은 이름으로 받는지)
!rm -rf /tmp/ours_unpacked && {PY} /tmp/inspect.py out/model.litertlm /tmp/ours_unpacked 2>&1 | grep -v Warning | tail -20
a = json.load(open("/tmp/official_unpacked/summary.json"))["sigs"]
b = json.load(open("/tmp/ours_unpacked/summary.json"))["sigs"]
for k in sorted(set(a) | set(b)):
    print(k, "같음" if a.get(k) == b.get(k) else f"다름\n  공식: {str(a.get(k))[:600]}\n  우리: {str(b.get(k))[:600]}")

In [ ]:
!timeout 900 litert-lm run out/model.litertlm --prompt="5분만 더 잘래" 2>&1 | tail -3 | cut -c1-400
!timeout 900 litert-lm run out/model.litertlm --prompt="목이 너무 말라" 2>&1 | tail -3 | cut -c1-400

### 점검: 우리 파일이 이상한 기호만 냄 → 공식 PLE 값의 배율이 우리 본체와 맞는지

In [ ]:
%%writefile /tmp/check_ple.py
# 공식 PLE tflite 출력이 우리 모델의 get_per_layer_inputs(= 표 * sqrt(d))와 같은지 숫자로 비교한다.
import sys, glob, json, numpy as np, torch
from safetensors import safe_open
from ai_edge_litert.interpreter import Interpreter
merged, ple = sys.argv[1], sys.argv[2]
cfg = json.load(open(merged + "/config.json")); tc = cfg.get("text_config", cfg)
d = tc["hidden_size_per_layer_input"]; print("d =", d, "layers =", tc["num_hidden_layers"])
it = Interpreter(model_path=ple); print(it.get_signature_list())
r = it.get_signature_runner("per_layer_embedder")
print("in:", [(x, v["shape"], v["dtype"]) for x, v in r.get_input_details().items()])
print("out:", [(x, v["shape"], v["dtype"]) for x, v in r.get_output_details().items()])
ids = [2, 818, 5000, 100000, 200000]
o = np.concatenate([np.asarray(list(r(token_ids=np.array([[t]], dtype=np.int32)).values())[0]).reshape(1, -1) for t in ids]).astype(np.float64)
e = None
for p in sorted(glob.glob(merged + "/*.safetensors")):
    with safe_open(p, "pt") as f:
        for k in f.keys():
            if k.endswith("embed_tokens_per_layer.weight") and "vision" not in k and "audio" not in k:
                sl = f.get_slice(k); e = torch.cat([sl[t:t+1] for t in ids]).float().numpy().astype(np.float64); print(k); break
    if e is not None: break
print("official out", o.shape, "table rows", e.shape)
for name, ref in [("raw", e), ("x sqrt(d)", e * np.sqrt(d))]:
    s = (o * ref).sum() / (ref * ref).sum()
    print(f"{name:10s}: best scale={s:.4f}  rel_err={np.linalg.norm(o - s * ref) / np.linalg.norm(o):.4f}")
print("official sample:", o[0, :6]); print("table*sqrt(d):", (e * np.sqrt(d))[0, :6])


In [ ]:
!{PY} /tmp/check_ple.py {MERGED} {OFFICIAL_PLE} 2>&1 | grep -v -E "Warning|wrapt|sitecustomize" | tail -15


In [ ]:
%%writefile /tmp/check_ple2.py
# 공식 PLE 출력과 표의 관계를 더 본다: 0 비율, 배치(35x256 vs 256x35), 층별 상관
import sys, glob, numpy as np, torch
from safetensors import safe_open
from ai_edge_litert.interpreter import Interpreter
merged, ple = sys.argv[1], sys.argv[2]
r = Interpreter(model_path=ple).get_signature_runner("per_layer_embedder")
ids = [818, 5000, 100000]
o = np.stack([np.asarray(r(token_ids=np.array([[t]], dtype=np.int32))["embeddings"]).reshape(-1) for t in ids]).astype(np.float64)
for p in sorted(glob.glob(merged + "/*.safetensors")):
    with safe_open(p, "pt") as f:
        ks = [k for k in f.keys() if k.endswith("embed_tokens_per_layer.weight") and "vision" not in k]
        if ks: sl = f.get_slice(ks[0]); e = torch.cat([sl[t:t+1] for t in ids]).float().numpy().astype(np.float64); break
def corr(a, b): a = a - a.mean(); b = b - b.mean(); return float((a*b).sum() / np.sqrt((a*a).sum() * (b*b).sum()))
print("0 비율(공식):", float((o == 0).mean()), " 표 0 비율:", float((e == 0).mean()))
print("공식 min/max/std:", o.min(), o.max(), o.std(), " 표*16 std:", (e*16).std())
print("corr 같은 배치:", corr(o, e), " 256x35 전치:", corr(o, np.stack([x.reshape(256, 35).T.reshape(-1) for x in e])))
L = [corr(o[:, i*256:(i+1)*256], e[:, i*256:(i+1)*256]) for i in range(35)]
print("층별 corr:", [round(x, 2) for x in L])
print("다른 토큰끼리 corr (공식0 vs 표1):", corr(o[0], e[1]))


In [ ]:
!{PY} /tmp/check_ple2.py {MERGED} {OFFICIAL_PLE} 2>&1 | grep -v -E "Warning|wrapt|sitecustomize|INFO" | tail -8


In [ ]:
# 대조군: 같은 기계에서 공식 모델은 제대로 대답하는지
!timeout 900 litert-lm run /tmp/official/gemma-4-E2B-it.litertlm --prompt="5분만 더 잘래" 2>&1 | tail -3 | cut -c1-400


### (확인용) 단어 표가 원본과 같은지 — 원본 10GB를 받아야 해서 맨 끝에 둔다

In [ ]:
%%writefile /tmp/verify_ple.py
# 미세조정 모델의 단어 표가 학습 전 원본과 같은지 비교한다 (메모리를 아끼려고 1만 행씩).
import sys, glob, torch
from safetensors import safe_open
from huggingface_hub import hf_hub_download
merged = sys.argv[1]
base = hf_hub_download("unsloth/gemma-4-E2B-it", "model.safetensors", local_dir="/tmp/base")
def find(paths, suffix):
    for p in paths:
        with safe_open(p, "pt") as f:
            for k in f.keys():
                if k.endswith(suffix) and "vision" not in k and "audio" not in k:
                    return p, k
    raise KeyError(suffix)
mp = sorted(glob.glob(merged + "/*.safetensors"))
for name in ["embed_tokens_per_layer.weight", "embed_tokens.weight"]:
    (pa, ka), (pb, kb) = find(mp, "." + name), find([base], "." + name)
    with safe_open(pa, "pt") as fa, safe_open(pb, "pt") as fb:
        a, b = fa.get_tensor(ka), fb.get_tensor(kb)
        same_bits = a.dtype == b.dtype and a.shape == b.shape and torch.equal(a, b)
        mx = 0.0
        if not same_bits and a.shape == b.shape:
            for i in range(0, a.shape[0], 10000):
                mx = max(mx, (a[i:i+10000].float() - b[i:i+10000].float()).abs().max().item())
        print(f"{name}: merged={ka} {tuple(a.shape)} {a.dtype} | base={kb} {tuple(b.shape)} {b.dtype}")
        print("   ->", "완전히 같음 (EQUAL)" if same_bits else f"다름: 최대 차이 {mx}")
        del a, b


In [ ]:
!{PY} /tmp/verify_ple.py {MERGED} 2>&1 | grep -v Warning | tail -8